In [6]:
## RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
# Add the document chunks to the "vector store" using OpenAIEmbeddings
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings,
)

# With langchain we can easily turn any vector store into a retrieval component:
retriever = vectorstore.as_retriever(k=6)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5864.96it/s]


In [7]:
retriever.invoke("what is agents")

[Document(id='82798f9d-a650-4e6a-9890-f77d58c45226', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, 

In [10]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

response = llm.invoke("What is RAG?")

print(response.content)

**RAG = Retrieval‑Augmented Generation**

Retrieval‑Augmented Generation (RAG) is a hybrid AI architecture that combines two complementary capabilities:

| Component | What it does | Why it’s useful |
|-----------|--------------|-----------------|
| **Retriever** | Searches an external knowledge source (e.g., a vector store, database, or the web) for the most relevant documents or passages given a user query. | Gives the model access to up‑to‑date, factual, and domain‑specific information that it may not have memorized during pre‑training. |
| **Generator** | A large language model (LLM) that takes the retrieved text (often concatenated with the original prompt) and produces a natural‑language response. | Allows the system to produce fluent, context‑aware answers, synthesize multiple sources, and follow conversational instructions. |

The overall flow is:

1. **User query** → (optional preprocessing)  
2. **Retriever** fetches *k* top‑ranked chunks from a knowledge base.  
3. Retrieved

In [11]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question:str)->dict:
    ## Relevant context
    docs=retriever.invoke(question)
    docs_string = " ".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
    
    ## llm invoke

    ai_msg=llm.invoke([
         {"role": "system", "content": instructions},
        {"role": "user", "content": question},

    ])
    return {"answer":ai_msg.content,"documents":docs}

In [12]:
rag_bot("What is agents")

{'answer': 'Agents are autonomous software entities that perceive their environment, maintain internal memory, and use a large language model as their “brain” to plan, decompose tasks, reflect on actions, and execute tool‑use or other behaviors. They coordinate with other agents and observations, leveraging hierarchical planning and memory retrieval to act believably over time. In LLM‑powered systems, these components together enable agents to solve complex problems without direct human supervision.',
 'documents': [Document(id='82798f9d-a650-4e6a-9890-f77d58c45226', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays 